In [2]:
from functools import reduce
from pyspark.sql import SparkSession, functions as F

In [3]:
spark = SparkSession.builder.appName("02_silver").getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

BUCKET = "taxis-chicago-grandes-datos-506301"
BRONZE_TAXI = f"gs://{BUCKET}/bronze/taxi_trips"
SILVER_TAXI = f"gs://{BUCKET}/silver/taxi_trips"
QUALITY_REPORT = f"gs://{BUCKET}/silver/_quality_report"

bronze = spark.read.parquet(BRONZE_TAXI)

26/10/04 17:56:30 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
# 7) Clima: solo la estación de Chicago, fecha, unidades métricas y valores faltantes
STATION = "stn = '725300' AND wban = '94846'"   # aeropuerto O'Hare
SILVER_WEATHER = f"gs://{BUCKET}/silver/weather"

weather_bronze = spark.read.option("mergeSchema", "true").parquet(f"gs://{BUCKET}/bronze/weather")
weather = (weather_bronze.where(STATION)
           .select(F.to_date(F.concat_ws("-", "year", "mo", "da")).alias("date"),
                   F.col("temp").cast("double").alias("temp_f"),
                   F.col("prcp").cast("double").alias("prcp_in"),
                   F.col("snow_ice_pellets").cast("int").alias("snow"))
           .withColumn("temp_c", F.round((F.when(F.col("temp_f") < 200, F.col("temp_f")) - 32) * 5 / 9, 1))
           .withColumn("prcp_mm", F.round(F.when(F.col("prcp_in") < 99, F.col("prcp_in")).otherwise(0.0) * 25.4, 1))
           .select("date", "temp_c", "prcp_mm", "snow")
           .dropDuplicates(["date"]))
weather.coalesce(1).write.mode("overwrite").parquet(SILVER_WEATHER)

print(f"Estaciones en bronze: {weather_bronze.select('stn', 'wban').distinct().count():,}")
print(f"Días de clima en silver: {spark.read.parquet(SILVER_WEATHER).count():,}")

Estaciones en bronze: 15,683
Días de clima en silver: 4,017


In [5]:
# 1) Columnas que se usan, con tipos explícitos y texto normalizado
df = bronze.select(
    "unique_key", "taxi_id",
    "trip_start_timestamp", "trip_end_timestamp",
    F.col("trip_seconds").cast("int").alias("trip_seconds"),
    F.col("trip_miles").cast("double").alias("trip_miles"),
    F.col("pickup_community_area").cast("int").alias("pickup_community_area"),
    F.col("dropoff_community_area").cast("int").alias("dropoff_community_area"),
    F.col("fare").cast("double").alias("fare"),
    F.col("tips").cast("double").alias("tips"),
    F.col("tolls").cast("double").alias("tolls"),
    F.col("extras").cast("double").alias("extras"),
    F.col("trip_total").cast("double").alias("trip_total"),
    F.initcap(F.trim("payment_type")).alias("payment_type"),
    F.trim("company").alias("company"),
    F.col("pickup_latitude").cast("double").alias("pickup_latitude"),
    F.col("pickup_longitude").cast("double").alias("pickup_longitude"),
)

In [6]:
# 2) Duplicados: unique_key NO es única en la fuente
dedup = df.dropDuplicates(["unique_key"])

In [7]:
# 3) Reglas de calidad: True = la fila se rechaza
rules = {
    "missing_start_time":  F.col("trip_start_timestamp").isNull(),
    "missing_pickup_area": F.col("pickup_community_area").isNull(),
    "invalid_duration":    F.col("trip_seconds").isNull() | ~F.col("trip_seconds").between(60, 4 * 3600),
    "invalid_fare":        F.col("fare").isNull() | (F.col("fare") <= 0) | (F.col("fare") > 500),
    "end_before_start":    F.col("trip_end_timestamp") < F.col("trip_start_timestamp"),
}
flagged = dedup
for name, condition in rules.items():
    flagged = flagged.withColumn(name, F.coalesce(condition, F.lit(False)))
flagged = flagged.withColumn("rejected", reduce(lambda a, b: a | b, [F.col(n) for n in rules]))

In [8]:
# 4) Reporte de calidad (una sola pasada sobre los datos)
bronze_rows = bronze.count()
counts = flagged.agg(
    F.count("*").alias("rows_without_duplicates"),
    *[F.sum(F.col(n).cast("int")).alias(n) for n in rules],
    F.sum(F.col("rejected").cast("int")).alias("rejected_total"),
).first().asDict()
counts["bronze_rows"] = bronze_rows
counts["duplicates"] = bronze_rows - counts["rows_without_duplicates"]
counts["silver_rows"] = counts["rows_without_duplicates"] - counts["rejected_total"]
report = spark.createDataFrame([(k, int(v)) for k, v in counts.items()], ["metric", "rows"])
report.show(truncate=False)
report.coalesce(1).write.mode("overwrite").parquet(QUALITY_REPORT)

+-----------------------+---------+
|metric                 |rows     |
+-----------------------+---------+
|rows_without_duplicates|169511471|
|missing_start_time     |0        |
|missing_pickup_area    |21587949 |
|invalid_duration       |14031713 |
|invalid_fare           |305605   |
|end_before_start       |443118   |
|rejected_total         |27178162 |
|bronze_rows            |212046737|
|duplicates             |42535266 |
|silver_rows            |142333309|
+-----------------------+---------+



In [9]:
# 5) Nulos que no se eliminan: se corrigen o se etiquetan
silver = (flagged.filter(~F.col("rejected"))
          .drop("rejected", *rules.keys())
          .withColumn("trip_miles", F.when(F.col("trip_miles") > 0, F.col("trip_miles")))  # 0 = taxímetro sin dato
          .na.fill({"tips": 0.0, "tolls": 0.0, "extras": 0.0,
                    "company": "Unknown", "payment_type": "Unknown"})
          .withColumn("trip_total", F.coalesce("trip_total",
                                               F.col("fare") + F.col("tips") + F.col("tolls") + F.col("extras")))
          .withColumn("trip_date", F.to_date("trip_start_timestamp"))
          .withColumn("trip_hour", F.hour("trip_start_timestamp"))
          .withColumn("year", F.year("trip_start_timestamp"))
          .withColumn("month", F.month("trip_start_timestamp")))

In [10]:
# 6) Escritura optimizada: Parquet particionado por año y mes
(silver.repartition("year", "month")
 .write.mode("overwrite").partitionBy("year", "month").parquet(SILVER_TAXI))

26/10/04 18:05:19 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [11]:
check = spark.read.parquet(SILVER_TAXI)
print(f"Filas en silver: {check.count():,}")
check.groupBy("year").count().orderBy("year").show()

Filas en silver: 142,333,309


+----+--------+
|year|   count|
+----+--------+
|2013|21692018|
|2014|30521190|
|2015|19597100|
|2016|15530763|
|2017|10313038|
|2018|11490599|
|2019|14790671|
|2020| 3462949|
|2021| 3445170|
|2022| 5554772|
|2023| 5935039|
+----+--------+



In [12]:
spark.stop()